# Semaine 2 — Jour 1 : Architecture d’un agent

Notebook étudiant.

## Objectifs

- Comprendre les composants d’un agent.
- Implémenter un agent minimal.
- Observer une trace d’exécution.

## Architecture conceptuelle

```mermaid
flowchart LR
    User[Utilisateur] --> Agent[Agent]
    Agent --> State[État]
    Agent --> Policy[Politique]
    Policy --> Tool[Outil]
    Tool --> Observation[Observation]
    Observation --> Agent
    Agent --> Answer[Réponse]
```

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass
class AgentState:
    """State container for a minimal mono-agent."""

    user_input: str
    action: str | None = None
    observation: dict[str, Any] | None = None
    final_answer: str | None = None
    trace: list[str] = field(default_factory=list)


def extract_order_id(text: str) -> str:
    """Extract the first numeric token from a text."""
    normalized = text.replace("?", " ").replace(".", " ")
    for token in normalized.split():
        if token.isdigit():
            return token
    return "unknown"


def extract_sku(text: str) -> str:
    """Extract a simple SKU-like token from a text."""
    normalized = text.replace("?", " ").replace(".", " ")
    for token in normalized.split():
        if token.upper().startswith("SKU-"):
            return token.upper()
    return "UNKNOWN"


def lookup_order(order_id: str) -> dict[str, str]:
    """Simulated order lookup tool."""
    return {
        "order_id": order_id,
        "status": "expédiée",
        "eta": "2026-07-05",
    }


def lookup_inventory(sku: str) -> dict[str, Any]:
    """Simulated inventory lookup tool."""
    return {
        "sku": sku,
        "available": True,
        "quantity": 12,
    }


def decide_action(user_input: str) -> str:
    """Select the next action using deterministic rules."""
    text = user_input.lower()

    if "commande" in text:
        return "lookup_order"

    if "produit" in text or "stock" in text or "sku-" in text:
        return "lookup_inventory"

    return "answer_directly"


def run_agent(user_input: str) -> AgentState:
    """Run a minimal mono-agent for one user request."""
    state = AgentState(user_input=user_input)
    state.trace.append("received_user_input")

    state.action = decide_action(user_input)
    state.trace.append(f"selected_action:{state.action}")

    if state.action == "lookup_order":
        order_id = extract_order_id(user_input)
        state.trace.append(f"extracted_order_id:{order_id}")

        state.observation = lookup_order(order_id)
        state.trace.append("called_tool:lookup_order")

        state.final_answer = (
            f"Votre commande {state.observation['order_id']} est "
            f"{state.observation['status']}. "
            f"Livraison estimée : {state.observation['eta']}."
        )
        state.trace.append("generated_final_answer")
        return state

    if state.action == "lookup_inventory":
        sku = extract_sku(user_input)
        state.trace.append(f"extracted_sku:{sku}")

        state.observation = lookup_inventory(sku)
        state.trace.append("called_tool:lookup_inventory")

        availability = "disponible" if state.observation["available"] else "indisponible"
        state.final_answer = (
            f"Le produit {state.observation['sku']} est {availability}. "
            f"Quantité disponible : {state.observation['quantity']}."
        )
        state.trace.append("generated_final_answer")
        return state

    state.final_answer = "Bonjour, je peux vous aider. Quelle est votre demande ?"
    state.trace.append("generated_final_answer")
    return state


def run_tests() -> None:
    order_state = run_agent("Où est ma commande 123 ?")
    assert order_state.action == "lookup_order"
    assert "commande 123" in order_state.final_answer
    assert "called_tool:lookup_order" in order_state.trace

    inventory_state = run_agent("Le produit SKU-42 est-il disponible ?")
    assert inventory_state.action == "lookup_inventory"
    assert "SKU-42" in inventory_state.final_answer
    assert "called_tool:lookup_inventory" in inventory_state.trace

    direct_state = run_agent("Bonjour")
    assert direct_state.action == "answer_directly"
    assert direct_state.observation is None
    assert "generated_final_answer" in direct_state.trace


if False:
    example = run_agent("Où est ma commande 123 ?")

    print("Final answer:")
    print(example.final_answer)
    print()

    print("Selected action:")
    print(example.action)
    print()

    print("Trace:")
    for event in example.trace:
        print(f"- {event}")

    run_tests()
    print()
    print("All tests passed.")

## À faire

Exécutez l’agent avec trois entrées :

1. `Où est ma commande 123 ?`
2. `Le produit SKU-42 est-il disponible ?`
3. `Bonjour`

Observez l’action sélectionnée, la réponse finale et la trace.

In [ ]:
examples = [
    "Où est ma commande 123 ?",
    "Le produit SKU-42 est-il disponible ?",
    "Bonjour",
]

for user_input in examples:
    state = run_agent(user_input)
    print("=" * 60)
    print("Input:", user_input)
    print("Action:", state.action)
    print("Answer:", state.final_answer)
    print("Trace:", state.trace)

## Questions

- Où se trouve la logique de décision ?
- Où se trouve l’état ?
- Quelle trace permet de prouver qu’un outil a été appelé ?
- Quelle limite principale voyez-vous dans cette version déterministe ?

# Corrections enseignant

# Corrigé — Exercices

## Exercice 1 — Identifier les composants

Scénario :

```text
Peux-tu vérifier la disponibilité du produit SKU-42 ?
```

Réponse possible :

| Élément | Réponse |
|---|---|
| Entrée utilisateur | `Peux-tu vérifier la disponibilité du produit SKU-42 ?` |
| Intention probable | Vérifier un stock produit |
| Outil nécessaire | `lookup_inventory` |
| Observation attendue | Disponibilité, quantité, SKU |
| Réponse finale possible | `Le produit SKU-42 est disponible. Quantité disponible : 12.` |

## Exercice 2 — Compléter une politique de décision

```python
def decide_action(user_input: str) -> str:
    text = user_input.lower()

    if "commande" in text:
        return "lookup_order"

    if "produit" in text or "stock" in text:
        return "lookup_inventory"

    return "answer_directly"
```

Une variante plus robuste peut aussi détecter `sku-` :

```python
def decide_action(user_input: str) -> str:
    text = user_input.lower()

    if "commande" in text:
        return "lookup_order"

    if "produit" in text or "stock" in text or "sku-" in text:
        return "lookup_inventory"

    return "answer_directly"
```

## Exercice 3 — Ajouter un outil simulé

```python
def lookup_inventory(sku: str) -> dict:
    return {
        "sku": sku,
        "available": True,
        "quantity": 12,
    }
```

## Exercice 4 — Ajouter une trace

Exemple d’intégration :

```python
state.trace.append("received_user_input")

state.action = decide_action(user_input)
state.trace.append(f"selected_action:{state.action}")

if state.action == "lookup_inventory":
    state.observation = lookup_inventory("SKU-42")
    state.trace.append("called_tool:lookup_inventory")

state.trace.append("generated_final_answer")
```

## Exercice 5 — Tester mentalement l’agent

| Entrée | Action attendue |
|---|---|
| `Où est ma commande 123 ?` | `lookup_order` |
| `Le produit SKU-42 est-il disponible ?` | `lookup_inventory` |
| `Bonjour` | `answer_directly` |

## Exercice 6 — Question de conception

Mélanger l’analyse d’intention, l’appel d’outil, la génération de réponse et la gestion d’état rend l’agent difficile à :

- tester ;
- déboguer ;
- faire évoluer ;
- observer ;
- sécuriser.

Une architecture agentique fiable sépare les responsabilités pour que chaque composant puisse être validé indépendamment.

# Corrigé — Questions d’entretien

## 1. Quelle différence fais-tu entre un modèle de langage et un agent ?

Un modèle de langage produit une sortie à partir d’une entrée.

Un agent est une architecture logicielle autour du modèle. Il peut maintenir un état, décider d’une action, appeler des outils, lire des observations et produire une réponse finale.

## 2. Pourquoi un agent doit-il maintenir un état ?

L’état permet de conserver les informations utiles pendant l’exécution :

- demande utilisateur ;
- action choisie ;
- observations ;
- réponse finale ;
- trace.

Sans état explicite, le comportement devient difficile à inspecter et à tester.

## 3. Qu’est-ce qu’un outil dans une architecture agentique ?

Un outil est une fonction ou un service externe que l’agent peut appeler pour obtenir une information ou effectuer une action.

Exemples :

- rechercher une commande ;
- consulter un stock ;
- interroger une base de données ;
- appeler une API interne.

## 4. À quoi sert une trace d’exécution ?

La trace d’exécution permet de comprendre le chemin suivi par l’agent.

Elle aide à :

- déboguer ;
- auditer ;
- évaluer ;
- expliquer ;
- tester.

## 5. Quelle est la différence entre mémoire et état ?

L’état concerne l’exécution courante.

La mémoire désigne des informations conservées au-delà d’une seule exécution ou conversation.

Exemple :

- état : action choisie pour la demande actuelle ;
- mémoire : préférence utilisateur sauvegardée pour les prochaines sessions.

## 6. Pourquoi séparer la politique de décision de l’exécution des outils ?

La séparation permet de tester la décision sans déclencher d’effet externe.

Elle évite aussi qu’un outil sensible soit appelé par erreur à cause d’une logique difficile à isoler.

## 7. Quels risques apparaissent lorsqu’un agent peut appeler des outils externes ?

Risques principaux :

- appels non désirés ;
- effets irréversibles ;
- fuite de données ;
- coûts inattendus ;
- erreurs silencieuses ;
- dépendance à des services externes.

## 8. Pourquoi commencer par un agent mono-agent ?

Un agent mono-agent est plus simple à comprendre, tester et observer.

Il permet de maîtriser les fondamentaux avant d’introduire la coordination multi-agent, qui ajoute de la complexité.

## Question de raisonnement 1

Pour rendre le comportement testable, je séparerais :

- `decide_action` ;
- `lookup_order` ;
- `lookup_inventory` ;
- `run_agent` ;
- `AgentState`.

Ensuite, j’écrirais des tests par chemin :

- demande commande ;
- demande stock ;
- demande générale.

## Question de raisonnement 2

Il faut inspecter :

1. la politique de décision ;
2. les conditions de déclenchement de l’outil ;
3. les garde-fous avant appel ;
4. la trace d’exécution ;
5. les tests couvrant les cas sensibles.

## Question de raisonnement 3

J’ajouterais :

- une trace structurée ;
- des logs par étape ;
- un identifiant de requête ;
- l’action sélectionnée ;
- les entrées/sorties des outils non sensibles ;
- les erreurs ;
- la latence ;
- le statut final.

# Corrigé — Challenge

## Solution complète

```python
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass
class AgentState:
    user_input: str
    action: str | None = None
    observation: dict[str, Any] | None = None
    final_answer: str | None = None
    trace: list[str] = field(default_factory=list)


def extract_order_id(text: str) -> str:
    normalized = text.replace("?", " ").replace(".", " ")
    for token in normalized.split():
        if token.isdigit():
            return token
    return "unknown"


def extract_sku(text: str) -> str:
    normalized = text.replace("?", " ").replace(".", " ")
    for token in normalized.split():
        if token.upper().startswith("SKU-"):
            return token.upper()
    return "UNKNOWN"


def lookup_order(order_id: str) -> dict[str, str]:
    return {
        "order_id": order_id,
        "status": "expédiée",
        "eta": "2026-07-05",
    }


def lookup_inventory(sku: str) -> dict[str, Any]:
    return {
        "sku": sku,
        "available": True,
        "quantity": 12,
    }


def decide_action(user_input: str) -> str:
    text = user_input.lower()

    if "commande" in text:
        return "lookup_order"

    if "produit" in text or "stock" in text or "sku-" in text:
        return "lookup_inventory"

    return "answer_directly"


def run_agent(user_input: str) -> AgentState:
    state = AgentState(user_input=user_input)
    state.trace.append("received_user_input")

    state.action = decide_action(user_input)
    state.trace.append(f"selected_action:{state.action}")

    if state.action == "lookup_order":
        order_id = extract_order_id(user_input)
        state.trace.append(f"extracted_order_id:{order_id}")
        state.observation = lookup_order(order_id)
        state.trace.append("called_tool:lookup_order")
        state.final_answer = (
            f"Votre commande {state.observation['order_id']} est "
            f"{state.observation['status']}. "
            f"Livraison estimée : {state.observation['eta']}."
        )
        state.trace.append("generated_final_answer")
        return state

    if state.action == "lookup_inventory":
        sku = extract_sku(user_input)
        state.trace.append(f"extracted_sku:{sku}")
        state.observation = lookup_inventory(sku)
        state.trace.append("called_tool:lookup_inventory")
        availability = "disponible" if state.observation["available"] else "indisponible"
        state.final_answer = (
            f"Le produit {state.observation['sku']} est {availability}. "
            f"Quantité disponible : {state.observation['quantity']}."
        )
        state.trace.append("generated_final_answer")
        return state

    state.final_answer = "Bonjour, je peux vous aider. Quelle est votre demande ?"
    state.trace.append("generated_final_answer")
    return state


def test_order_request() -> None:
    state = run_agent("Où est ma commande 123 ?")
    assert state.action == "lookup_order"
    assert state.observation["order_id"] == "123"
    assert "called_tool:lookup_order" in state.trace


def test_inventory_request() -> None:
    state = run_agent("Le produit SKU-42 est-il disponible ?")
    assert state.action == "lookup_inventory"
    assert state.observation["sku"] == "SKU-42"
    assert "called_tool:lookup_inventory" in state.trace


def test_direct_request() -> None:
    state = run_agent("Bonjour")
    assert state.action == "answer_directly"
    assert state.observation is None
    assert state.final_answer is not None


if __name__ == "__main__":
    test_order_request()
    test_inventory_request()
    test_direct_request()
    print("All tests passed.")
```

## Points importants

Cette solution reste volontairement déterministe.

Elle prépare les journées suivantes, où la sélection d’action sera progressivement remplacée ou augmentée par :

- function calling ;
- sorties structurées ;
- gestion d’état ;
- mémoire ;
- boucle agentique.

# Review — Week 02 Day 01

## Synthèse pédagogique

Cette journée introduit l’architecture agentique comme un problème d’ingénierie logicielle.

Le point central est la séparation des responsabilités :

- état ;
- décision ;
- outil ;
- observation ;
- réponse ;
- trace.

## Ce que l’apprenant doit retenir

Un agent n’est pas simplement un prompt plus long.

Un agent est une orchestration structurée autour d’un modèle ou d’une logique de décision.

## Signaux de compréhension

L’apprenant est capable de :

- dessiner une architecture d’agent ;
- expliquer le rôle de chaque composant ;
- coder un agent minimal ;
- lire une trace d’exécution ;
- identifier les limites d’un agent déterministe.

## Erreurs fréquentes

### Erreur 1 — Tout mettre dans une seule fonction

Cela rend le système difficile à tester et à faire évoluer.

### Erreur 2 — Confondre état et mémoire

L’état concerne l’exécution courante.

La mémoire persiste au-delà de l’exécution courante.

### Erreur 3 — Appeler les outils sans garde-fou

Un agent qui peut déclencher des outils externes doit être contrôlé.

### Erreur 4 — Ne pas tracer les décisions

Sans trace, l’agent devient opaque.

## Préparation du jour 2

Le jour 2 remplacera progressivement une décision codée à la main par du function calling.

Le contrat d’architecture reste le même :

```text
input -> state -> decision -> tool -> observation -> answer
```

## Critères de validation de la journée

- Tous les fichiers Markdown obligatoires existent.
- Les corrections sont locales.
- Le lab est exécutable.
- Le notebook étudiant exclut les corrections.
- Le notebook enseignant inclut les corrections.
- Les diagrammes sont présents.